In [2]:
# Library installations and imports for the project
!pip install ydata-profiling nltk spacy
!python -m spacy download en_core_web_sm
!pip install gensim
!pip install xgboost

import pandas as pd
import re
import nltk
import spacy
import numpy as np
import gensim
from ydata_profiling import ProfileReport
from nltk.corpus import stopwords
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

# Stopwords download
nltk.download('stopwords')
# spaCy initialization
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.5/682.5 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 73.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 60.6 MB/s eta 0:00:00


/tmp/ipykernel_521/731596251.py:13: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


**1. Preparing Data for the model**

 - Dataframe preparation

In [4]:
# Reading CSVs
true_df = pd.read_csv('/content/True.csv')
fake_df = pd.read_csv('/content/Fake.csv')

# Adding 'is_fake' columns
true_df['is_fake'] = 0
fake_df['is_fake'] = 1

# Combining the two CSVs into one Dataframe
df = pd.concat([true_df, fake_df], axis=0).reset_index(drop=True)

- Profile Report

In [5]:
# Profile Report generation
profile = ProfileReport(df, title="Fake News Profiling Report", explorative=True)
profile.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 5/5 [00:30<00:00,  6.02s/it]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

- Counting articles that have the word 'politics' in it's Category name



In [6]:
politics_count = df['subject'].str.contains('politics', case=False).sum()
print(f"Number of articles that have the word 'politics' in it's category name: {politics_count}")

Number of articles that have the word 'politics' in it's category name: 18113


- Removing columns we won't be needing for model training and saving the file

In [7]:
df = df.drop(columns=['subject', 'date'])

- Data cleaning and normalization

In [4]:
# Creating a column with both the article title and the text
df['full_text'] = df['title'] + " " + df['text']

# Setting stop words
stop_words = set(stopwords.words('english'))

# Defining a function that cleans the text
def clean_text(text):
    # Converting to a lower case letters
    text = text.lower()
    # Removing everything that's not a letter or a space
    text = re.sub(r'[^a-z\s]', '', text)
    # Removing extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Applying the clean_text function to the full_text column
df['full_text'] = df['full_text'].apply(clean_text)

# Defining a function to filter stop words and lemmatize the text
def filter_and_lemmatize(texts):
    processed_texts = []
    # Using nlp.pipe for batch processing of the articles
    for doc in nlp.pipe(texts, batch_size=500, n_process=-1):
        # Filtering out stop words and lemmatization
        lemmas = [token.lemma_ for token in doc if token.text not in stop_words and len(token.lemma_) > 1]
        processed_texts.append(" ".join(lemmas))
    return processed_texts

# Applying the filter and lemmatization function to the full_text column and saving the result in final_processed_text column
df['final_processed_text'] = filter_and_lemmatize(df['full_text'].tolist())


 - Saving the data frame as csv for backukp

In [7]:
df.to_csv('/content/news_dataset_preprocessed.csv', index=False)

**2. Training Models**

- Training Word2Vec model

In [9]:
# Loading the csv backup if needed
#df = pd.read_csv('/content/news_dataset_preprocessed.csv')

In [10]:
# Spliting texts to train and test
train_texts, test_texts, y_train, y_test = train_test_split(
    df['final_processed_text'].tolist(),
    df['is_fake'].values,
    test_size=0.2,
    random_state=42,
    stratify=df['is_fake'].values
)

# Tokenization of both train and test texts
tokenized_train = [text.split() for text in train_texts]
tokenized_test = [text.split() for text in test_texts]

# Word2Vec model training on just the train texts
w2v_model = Word2Vec(
    sentences=tokenized_train,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=10
)

 - Calculating averaged Word2Vec vector

In [11]:
def averaged_w2v(tokens, model, vector_size):
    # Filtering just the words Word2Vec learned
    valid_vectors = [model.wv[word] for word in tokens if word in model.wv]

    # Calculatin mean for valid vectors otherwise return zeros
    if valid_vectors:
        return np.mean(valid_vectors, axis=0)
    else:
        return np.zeros(vector_size)

# Vectorization of both X_train and X_test
X_train = np.array([averaged_w2v(tokens, w2v_model, 100) for tokens in tokenized_train])
X_test = np.array([averaged_w2v(tokens, w2v_model, 100) for tokens in tokenized_test])

 - Training Logistic Regression, Random Forest and XGBoost models

In [12]:
# Setting up the models in a dictionary
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss')
}

# Training the models and printing results
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"{name} results")
    print("-" * 50)
    print(classification_report(y_test, y_pred, target_names=['Real News (0)', 'Fake News (1)'], digits=4))
    print("\n")

Logistic Regression results
--------------------------------------------------
               precision    recall  f1-score   support

Real News (0)     0.9666    0.9718    0.9692      4284
Fake News (1)     0.9741    0.9693    0.9717      4696

     accuracy                         0.9705      8980
    macro avg     0.9703    0.9705    0.9704      8980
 weighted avg     0.9705    0.9705    0.9705      8980



Random Forest results
--------------------------------------------------
               precision    recall  f1-score   support

Real News (0)     0.9707    0.9596    0.9651      4284
Fake News (1)     0.9635    0.9736    0.9685      4696

     accuracy                         0.9669      8980
    macro avg     0.9671    0.9666    0.9668      8980
 weighted avg     0.9670    0.9669    0.9669      8980



XGBoost results
--------------------------------------------------
               precision    recall  f1-score   support

Real News (0)     0.9784    0.9734    0.9759      4284


**3. Model validation and performance evaluation**

In [13]:
from sklearn.model_selection import GridSearchCV

# Setting dictionaties for the grids
lr_grid = {
    'C': [0.1, 1.0, 10.0],
    'solver': ['lbfgs', 'liblinear']
}

rf_grid = {
    'n_estimators': [50, 100],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}

xgb_grid = {
    'n_estimators':[50, 100],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [3, 6, 9]
}

# Setting up the models in a dictionary
models_and_grids = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=42), lr_grid),
    'Random Forest': (RandomForestClassifier(random_state=42), rf_grid),
    'XGBoost': (XGBClassifier(random_state=42, eval_metric='logloss'), xgb_grid)
}

# Setting up variables for best hyperparameters and final predictions
best_hyperparameters = {}
final_predictions = {}

# Training the models to find the best hyperparameters and storing them in a variable and printing them
for name, (model, grid) in models_and_grids.items():
    grid_search = GridSearchCV(estimator=model, param_grid=grid, cv=3, scoring='accuracy', n_jobs=-1)
    grid_search.fit(X_train, y_train)

    best_hyperparameters[name] = grid_search.best_params_
    print(f"Best parameters for {name}: {grid_search.best_params_}\n")

# Training the models with the best hyperparameters
for name, (model, _) in models_and_grids.items():
    best_model = model.set_params(**best_hyperparameters[name])
    best_model.fit(X_train, y_train)
    final_predictions[name] = best_model.predict(X_test)

# Printing the updated reults
for name in models_and_grids.keys():
    print(f"{name} updated results")
    print("-" * 50)
    print(classification_report(y_test, final_predictions[name], target_names=['Real News (0)', 'Fake News (1)'], digits=4))
    print("\n")

Best parameters for Logistic Regression: {'C': 10.0, 'solver': 'lbfgs'}

Best parameters for Random Forest: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100}

Best parameters for XGBoost: {'learning_rate': 0.2, 'max_depth': 6, 'n_estimators': 100}

Logistic Regression updated results
--------------------------------------------------
               precision    recall  f1-score   support

Real News (0)     0.9673    0.9736    0.9705      4284
Fake News (1)     0.9758    0.9700    0.9729      4696

     accuracy                         0.9717      8980
    macro avg     0.9715    0.9718    0.9717      8980
 weighted avg     0.9717    0.9717    0.9717      8980



Random Forest updated results
--------------------------------------------------
               precision    recall  f1-score   support

Real News (0)     0.9711    0.9561    0.9635      4284
Fake News (1)     0.9605    0.9740    0.9672      4696

     accuracy                         0.9655      8980
    macro a

*When going through the dataset initially, I noticed that all the true news are from Reuters agency and have the name in the title. I thought that the models might learn to recognize this and that they might struggle, if Reuters released an article that was not true or other agency released fake news article mentioning Reuters in it. In the next part I will try to exclude the word "Reuters" from the model and compare the results.*

**4. Analysing influence of the "Reuters" word**

In [14]:
import re

# Defining a function that removes the word (Reuters) from the title
def remove_reuters(text):
    clean = re.sub(r'\breuters\b', '', text, flags=re.IGNORECASE)
    clean = re.sub(r'\s+', ' ', clean).strip()
    return clean

# Applying the function to the train and test splits
train_texts_no_reuters = [remove_reuters(text) for text in train_texts]
test_texts_no_reuters = [remove_reuters(text) for text in test_texts]

# Tokenizing the splits
tokenized_train_no_reuters = [text.split() for text in train_texts_no_reuters]
tokenized_test_no_reuters = [text.split() for text in test_texts_no_reuters]

# Learning Word2Vec without the word Reuters
w2v_model_no_reuters = Word2Vec(
    sentences=tokenized_train_no_reuters,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=10
)

# Converting to arrays
X_train_no_reuters = np.array([averaged_w2v(tokens, w2v_model_no_reuters, 100) for tokens in tokenized_train_no_reuters])
X_test_no_reuters = np.array([averaged_w2v(tokens, w2v_model_no_reuters, 100) for tokens in tokenized_test_no_reuters])

 - Training the models

In [15]:

no_reuters_test_predictions = {}


for name, (model, _) in models_and_grids.items():
    # Training the models with the best parameters again
    best_model_no_reuters = model.set_params(**best_hyperparameters[name])
    best_model_no_reuters.fit(X_train_no_reuters, y_train)

    preds_no_reuters = best_model_no_reuters.predict(X_test_no_reuters)
    no_reuters_test_predictions[name] = preds_no_reuters

    print(f"{name} results (no Reuters)")
    print("-" * 50)
    print(classification_report(y_test, preds_no_reuters, target_names=['Real News (0)', 'Fake News (1)'], digits=4))
    print("\n")

Logistic Regression results (no Reuters)
--------------------------------------------------
               precision    recall  f1-score   support

Real News (0)     0.9587    0.9634    0.9610      4284
Fake News (1)     0.9664    0.9621    0.9643      4696

     accuracy                         0.9627      8980
    macro avg     0.9625    0.9627    0.9626      8980
 weighted avg     0.9627    0.9627    0.9627      8980



Random Forest results (no Reuters)
--------------------------------------------------
               precision    recall  f1-score   support

Real News (0)     0.9670    0.9580    0.9625      4284
Fake News (1)     0.9620    0.9702    0.9661      4696

     accuracy                         0.9644      8980
    macro avg     0.9645    0.9641    0.9643      8980
 weighted avg     0.9644    0.9644    0.9644      8980



XGBoost results (no Reuters)
--------------------------------------------------
               precision    recall  f1-score   support

Real News (0)   

*There is a small decrease in the reports that I expected, but I thought that the difference might be bigger. Still, I think that having only one source for true news is a big problem. It is possible that the models learned the editorial style of Reuters, might still struggle if Reuters released an article that was false and might also struggle with true articles from other agencies, claiming them as fake news. To make the model better in real world, we would need to extend the dataset with true articles from other agencies.*

5. Comparing Training and Testing accuracy to check for overfitting and interpreting results

In [17]:
from sklearn.metrics import accuracy_score

print("Overfitting analysis\n")
for name, (model, _) in models_and_grids.items():
    # Training the models with best hyperprameters
    best_model_no_reuters = model.set_params(**best_hyperparameters[name])
    best_model_no_reuters.fit(X_train_no_reuters, y_train)

    # Prediction
    train_preds_no_reuters = best_model_no_reuters.predict(X_train_no_reuters)
    test_preds_no_reuters = best_model_no_reuters.predict(X_test_no_reuters)

    train_acc = accuracy_score(y_train, train_preds_no_reuters)
    test_acc = accuracy_score(y_test, test_preds_no_reuters)
    difference = train_acc - test_acc

    print(f"{name} (no Reuters):")
    print(f"  Training Accuracy: {train_acc:.4f}")
    print(f"  Testing Accuracy:  {test_acc:.4f}")
    print(f"  Difference:        {difference:.4f}")
    print("-" * 50)

Overfitting analysis

Logistic Regression (no Reuters):
  Training Accuracy: 0.9651
  Testing Accuracy:  0.9627
  Difference:        0.0024
--------------------------------------------------
Random Forest (no Reuters):
  Training Accuracy: 0.9996
  Testing Accuracy:  0.9644
  Difference:        0.0353
--------------------------------------------------
XGBoost (no Reuters):
  Training Accuracy: 0.9996
  Testing Accuracy:  0.9748
  Difference:        0.0247
--------------------------------------------------


*By changing the hyperparameters we managed to improve both XGboost precision and Logistic Regression precision, but not by very much. Overall in our project the XGBoost model performed with the best precision, however by comparing training and testing accuracy, it seems also a bit overfitted and there should be a room for improvement by changing it's hyperparameters. I also think, like I mentioned before, that having only one source for true news would make the model in it's current configuration very unreliable in real world, I think that it would be better suited as a model that recognizes articles written by Reuters rather than fake news.*